In [ ]:
import pygame
import sys
import os
import json
import time
import tkinter as tk
from tkinter import filedialog
from pygame.locals import *

%run logic.ipynb

%run gamerecorder.ipynb

class SOSGameUI:
    def __init__(self, game):
        self.player_type = {'Blue': 'human', 'Red': 'human'}
        self.game = game
        self.game.player_type = self.player_type
        
        self.recorder = GameRecorder(self.game)
        
        self.screen_width = 700
        self.screen_height = 700
        self.grid_size = 400
        self.start_x = 250
        self.start_y = 100
        
        self.sidebar_width = 220
        self.sidebar_padding = 20
        self.section_spacing = 30
        
        pygame.init()
        
        self.screen = pygame.display.set_mode((self.screen_width, self.screen_height))
        pygame.display.set_caption("SOS Game")

        self.BLACK = (0, 0, 0)
        self.WHITE = (255, 255, 255)
        self.GRAY = (200, 200, 200)
        self.LIGHT_GRAY = (230, 230, 230)
        self.BLUE = (0, 0, 255)
        self.RED = (255, 0, 0)
        self.LIGHT_BLUE = (173, 216, 230)
        self.GREEN = (0, 180, 0)
        self.YELLOW = (255, 255, 0)
        self.SECTION_HEADER_BG = (240, 240, 250)

        self.running = True
        
        # PIB_________
        try:
            self.font = pygame.font.SysFont("Arial", 24)
            self.small_font = pygame.font.SysFont("Arial", 20)
            self.large_font = pygame.font.SysFont("Arial", 32)
        except:
            self.font = pygame.font.Font(None, 24)
            self.small_font = pygame.font.Font(None, 20)
            self.large_font = pygame.font.Font(None, 32)
        
        self.button_width = 180
        self.button_height = 40
        self.button_margin = 15
        
        self.input_box = pygame.Rect(self.sidebar_padding, 0, 100, self.button_height)
        self.input_text = str(self.game.n)
        self.input_active = False
        self.buttons = []
        
        self.root = None
        
        replay_width = 120
        replay_height = 40
        replay_x = self.screen_width - replay_width - 20
        replay_y = 20
        self.replay_rect = pygame.Rect(replay_x, replay_y, replay_width, replay_height)
        
        self.create_layout()
        
        self.debug_buttons()
        
        if self.game.player_type[self.game.current_player] == 'computer' and not self.game.game_over:
            pygame.time.set_timer(pygame.USEREVENT, 1000)

    def create_layout(self):
        sidebar_x = self.sidebar_padding
        current_y = 20 
        
        self.section_headers = []
        self.section_headers.append(("Player Information", current_y))
        current_y += 35
        
        current_y += 60
        
        self.section_headers.append(("Game Mode", current_y))
        current_y += 35
        
        self.add_button(sidebar_x, current_y, self.button_width, self.button_height, "Simple Mode", self.set_simple_mode)
        current_y += self.button_height + self.button_margin
        self.add_button(sidebar_x, current_y, self.button_width, self.button_height, "General Mode", self.set_general_mode)
        current_y += self.button_height + self.section_spacing
        
        self.section_headers.append(("Board Size", current_y))
        current_y += 35
        
        self.input_box.y = current_y + 5
        self.input_box.width = 80
        refresh_x = self.input_box.x + self.input_box.width + 15
        self.add_button(refresh_x, current_y, 85, self.button_height, "Refresh", self.update_board_size)
        current_y += self.button_height + self.section_spacing
        
        self.section_headers.append(("Symbol Assignment", current_y))
        current_y += 35
        self.add_button(sidebar_x, current_y, self.button_width, self.button_height, "Swap S/O", self.swap_roles)
        current_y += self.button_height + self.section_spacing
        
        self.section_headers.append(("Player Types", current_y))
        current_y += 35
        
        self.add_button(sidebar_x, current_y, self.button_width, self.button_height, 
                       "Blue: Human", self.set_blue_human)
        current_y += self.button_height + 5
        
        self.add_button(sidebar_x, current_y, self.button_width, self.button_height, 
                       "Blue: Computer", self.set_blue_computer)
        current_y += self.button_height + 15
        
        self.add_button(sidebar_x, current_y, self.button_width, self.button_height, 
                       "Red: Human", self.set_red_human)
        current_y += self.button_height + 5
        
        self.add_button(sidebar_x, current_y, self.button_width, self.button_height, 
                       "Red: Computer", self.set_red_computer)
        current_y += self.button_height + self.section_spacing
        
        self.section_headers.append(("Recording", current_y))
        current_y += 35
        
        button_width_half = (self.button_width - 10) // 2
        self.add_button(sidebar_x, current_y, button_width_half, self.button_height, 
                       "Record", self.start_recording)
        self.add_button(sidebar_x + button_width_half + 10, current_y, button_width_half, self.button_height, 
                       "Stop", self.stop_recording)
        current_y += self.button_height + 10
        
        self.add_button(sidebar_x, current_y, button_width_half, self.button_height, 
                       "Save", self.save_game)
        self.add_button(sidebar_x + button_width_half + 10, current_y, button_width_half, self.button_height, 
                       "Load", self.load_game)
        current_y += self.button_height + 10
        
        self.section_headers.append(("Current Turn", current_y))
        self.current_turn_y = current_y + 35
        current_y += 70

        self.section_headers.append(("Scores", current_y))
        self.scores_y = current_y + 35
        current_y += 70
        
        self.recording_status_y = current_y
        current_y += 35

    def debug_buttons(self):
        print("\nDEBUG: Button Information:")
        for i, (rect, text, action) in enumerate(self.buttons):
            print(f"Button {i+1}: '{text}' at position ({rect.x}, {rect.y}), size ({rect.width}, {rect.height})")
            print(f"  Action: {action.__name__}")
        print(f"Replay Button: at position ({self.replay_rect.x}, {self.replay_rect.y}), size ({self.replay_rect.width}, {self.replay_rect.height})")
        print("\n")

    def draw_grid(self):
        self.cell_size = self.grid_size // self.game.n
        for row in range(1, self.game.n):
            pygame.draw.line(self.screen, self.BLACK, (self.start_x + row * self.cell_size, self.start_y),
                             (self.start_x + row * self.cell_size, self.start_y + self.grid_size), 2)
            pygame.draw.line(self.screen, self.BLACK, (self.start_x, self.start_y + row * self.cell_size),
                             (self.start_x + self.grid_size, self.start_y + row * self.cell_size), 2)
        pygame.draw.rect(self.screen, self.BLACK, (self.start_x, self.start_y, self.grid_size, self.grid_size), 2)

    def draw_game_status(self):
        if self.game.game_over:
            status_rect = pygame.Rect(self.start_x, self.start_y + self.grid_size + 20, 
                                     self.grid_size, 40)
            pygame.draw.rect(self.screen, self.LIGHT_GRAY, status_rect)
            pygame.draw.rect(self.screen, self.BLACK, status_rect, 2)
            
            if self.game.winner == 'Draw':
                text = "It's a draw!"
                color = self.BLACK
            else:
                text = f"{self.game.winner} wins!"
                color = self.RED if self.game.winner == 'Red' else self.BLUE
            
            game_over_text = self.large_font.render(text, True, color)
            text_x = self.start_x + (self.grid_size - game_over_text.get_width()) // 2
            text_y = self.start_y + self.grid_size + 25
            self.screen.blit(game_over_text, (text_x, text_y))
            
    def draw_replay_button(self):
        mouse_pos = pygame.mouse.get_pos()
        
        if self.replay_rect.collidepoint(mouse_pos):

            button_color = self.LIGHT_GRAY
        else:
            button_color = self.WHITE
            
        pygame.draw.rect(self.screen, button_color, self.replay_rect)
        pygame.draw.rect(self.screen, self.BLACK, self.replay_rect, 2)
        
        label = self.font.render("REPLAY", True, self.BLACK)
        self.screen.blit(label, (self.replay_rect.x + (self.replay_rect.width - label.get_width()) // 2, 
                                self.replay_rect.y + (self.replay_rect.height - label.get_height()) // 2))

    def draw_sidebar(self):
        pygame.draw.rect(self.screen, self.GRAY, (0, 0, self.sidebar_width, self.screen_height))
        
        for header_text, y_pos in self.section_headers:
            header_rect = pygame.Rect(0, y_pos - 5, self.sidebar_width, 30)
            pygame.draw.rect(self.screen, self.SECTION_HEADER_BG, header_rect)
            pygame.draw.line(self.screen, self.LIGHT_BLUE, (0, y_pos + 25), (self.sidebar_width, y_pos + 25), 2)

            self.draw_text(header_text, self.sidebar_padding, y_pos, self.BLACK, bold=True)
        
        blue_label = f"Blue: {self.player_type['Blue'].capitalize()} ({self.game.player_symbols['Blue']})"
        red_label = f"Red: {self.player_type['Red'].capitalize()} ({self.game.player_symbols['Red']})"
        self.draw_text(blue_label, self.sidebar_padding, self.section_headers[0][1] + 35, self.BLUE)
        self.draw_text(red_label, self.sidebar_padding, self.section_headers[0][1] + 65, self.RED)
        
        pygame.draw.rect(self.screen, self.WHITE, self.input_box)
        pygame.draw.rect(self.screen, self.BLACK, self.input_box, 2)
        text_surface = self.font.render(self.input_text, True, self.BLACK)
        self.screen.blit(text_surface, (self.input_box.x + 5, self.input_box.y + 5))
        
        color = self.BLUE if self.game.current_player == 'Blue' else self.RED
        self.draw_text(f"{self.game.current_player} ({self.game.player_symbols[self.game.current_player]})", 
                      self.sidebar_padding, self.current_turn_y, color)
        
        if self.game.mode == "General":
            if hasattr(self.game.game_instance, 'scores'):
                score_rect = pygame.Rect(self.sidebar_padding - 5, self.scores_y - 5,
                                        self.sidebar_width - 2*self.sidebar_padding + 10, 40)
                pygame.draw.rect(self.screen, self.WHITE, score_rect)
                pygame.draw.rect(self.screen, self.BLACK, score_rect, 2)
                
                blue_score = f"Blue: {self.game.game_instance.scores['Blue']}"
                red_score = f"Red: {self.game.game_instance.scores['Red']}"
                self.draw_text(blue_score, self.sidebar_padding + 5, self.scores_y, self.BLUE)
                self.draw_text(red_score, self.sidebar_padding + 100, self.scores_y, self.RED)

        if hasattr(self, 'recording_status_y'):
            status_text = ""
            status_color = self.BLACK
            
            if self.recorder.is_recording:
                status_text = "● Recording"
                status_color = self.RED
            elif self.recorder.is_replaying:
                status_text = "▶ Replaying"
                status_color = self.GREEN
            elif self.recorder.moves_history:
                status_text = "Game loaded"
                status_color = self.BLUE
                
            if status_text:
                self.draw_text(status_text, self.sidebar_padding, self.recording_status_y, status_color)

        mouse_pos = pygame.mouse.get_pos()
        for i, (rect, text, action) in enumerate(self.buttons):
            is_active_button = False
            
            if text == "Blue: Human" and self.player_type['Blue'] == 'human':
                is_active_button = True
            elif text == "Blue: Computer" and self.player_type['Blue'] == 'computer':
                is_active_button = True
            elif text == "Red: Human" and self.player_type['Red'] == 'human':
                is_active_button = True
            elif text == "Red: Computer" and self.player_type['Red'] == 'computer':
                is_active_button = True
            elif text == "Simple Mode" and self.game.mode == "Simple":
                is_active_button = True
            elif text == "General Mode" and self.game.mode == "General":
                is_active_button = True
            elif text == "Record" and self.recorder.is_recording:
                is_active_button = True
            
            if is_active_button:
                pygame.draw.rect(self.screen, self.LIGHT_BLUE, rect)
            elif rect.collidepoint(mouse_pos):
                pygame.draw.rect(self.screen, self.LIGHT_GRAY, rect)
            else:
                pygame.draw.rect(self.screen, self.WHITE, rect)
            
            pygame.draw.rect(self.screen, self.BLACK, rect, 2)
            label = self.font.render(text, True, self.BLACK)
            self.screen.blit(label, (rect.x + (rect.width - label.get_width()) // 2, rect.y + (rect.height - label.get_height()) // 2))

    def add_button(self, x, y, width, height, text, action):
        rect = pygame.Rect(x, y, width, height)
        self.buttons.append((rect, text, action))

    def draw_text(self, text, x, y, color, bold=False):
        if bold:
            temp_font = pygame.font.SysFont("Arial", 24, bold=True)
            label = temp_font.render(text, True, color)
        else:
            label = self.font.render(text, True, color)
        self.screen.blit(label, (x, y))

    def draw_board(self):
        self.cell_size = self.grid_size // self.game.n

        for row in range(self.game.n):
            for col in range(self.game.n):
                cell = self.game.board[row][col]
                if cell:
                    symbol, player = cell
                    color = self.BLUE if player == 'Blue' else self.RED
                    text = self.font.render(symbol, True, color)
                    x = self.start_x + col * self.cell_size + (self.cell_size - text.get_width()) // 2
                    y = self.start_y + row * self.cell_size + (self.cell_size - text.get_height()) // 2
                    self.screen.blit(text, (x, y))

    def update_board_size(self):
        if self.input_text.isdigit():
            size = int(self.input_text)
            if size > 2:
                new_game = SOSGame(size, self.game.mode, player_type=self.player_type)
                new_game.player_type = self.player_type
                self.game = new_game
                self.recorder = GameRecorder(self.game)
                self.refresh_ui()
        if self.game.player_type[self.game.current_player] == 'computer' and not self.game.game_over:
            pygame.time.set_timer(pygame.USEREVENT, 1000)

    def refresh_ui(self):
        self.screen.fill(self.WHITE)
        self.cell_size = self.grid_size // self.game.n
        self.draw_sidebar()
        self.draw_grid()
        self.draw_board()
        self.draw_game_status()
        self.draw_replay_button()
        pygame.display.flip()

    def set_blue_human(self):
        self.player_type['Blue'] = 'human'
        self.game.player_type = self.player_type
        self.refresh_ui()
        
        if self.game.current_player == 'Blue':
            pygame.time.set_timer(pygame.USEREVENT, 0)

    def set_blue_computer(self):
        self.player_type['Blue'] = 'computer'
        self.game.player_type = self.player_type
        self.refresh_ui()
        if self.game.current_player == 'Blue' and not self.game.game_over:
            pygame.time.set_timer(pygame.USEREVENT, 1000)

    def set_red_human(self):
        self.player_type['Red'] = 'human'
        self.game.player_type = self.player_type
        self.refresh_ui()
        if self.game.current_player == 'Red':
            pygame.time.set_timer(pygame.USEREVENT, 0)

    def set_red_computer(self):
        self.player_type['Red'] = 'computer'
        self.game.player_type = self.player_type
        self.refresh_ui()
        if self.game.current_player == 'Red' and not self.game.game_over:
            pygame.time.set_timer(pygame.USEREVENT, 1000)

    def set_simple_mode(self):
        new_game = SOSGame(self.game.n, mode="Simple", player_type=self.player_type)
        new_game.player_type = self.player_type
        self.game = new_game
        self.recorder = GameRecorder(self.game)
        
        if self.game.player_type[self.game.current_player] == 'computer' and not self.game.game_over:
            pygame.time.set_timer(pygame.USEREVENT, 200)
        else:
            pygame.time.set_timer(pygame.USEREVENT, 0)
            
        self.refresh_ui()

    def set_general_mode(self):
        new_game = SOSGame(self.game.n, mode="General", player_type=self.player_type)
        new_game.player_type = self.player_type
        new_game.game_instance.scores = {'Blue': 0, 'Red': 0}
        self.game = new_game
        self.recorder = GameRecorder(self.game)
        
        if self.game.player_type[self.game.current_player] == 'computer' and not self.game.game_over:
            pygame.time.set_timer(pygame.USEREVENT, 200)
        else:
            pygame.time.set_timer(pygame.USEREVENT, 0)
            
        self.refresh_ui()

    def swap_roles(self):
        self.game.swap_roles()
        self.refresh_ui()

    def start_recording(self):
        print("Starting recording...")
        self.recorder.start_recording()
        self.refresh_ui()
    
    def stop_recording(self):
        print("Stopping recording...")
        self.recorder.stop_recording()
        self.refresh_ui()
    
    def save_game(self):
        if not self.recorder.moves_history:
            print("No moves to save!")
            return
        
        if not self.root:
            self.root = tk.Tk()
            self.root.withdraw()
            
        try:
            file_path = filedialog.asksaveasfilename(
                defaultextension=".sos",
                filetypes=[("SOS Game Files", "*.sos"), ("Text Files", "*.txt"), ("All Files", "*.*")],
                title="Save Game Recording"
            )
            
            if file_path:
                self.recorder.save_to_file(file_path)
                self.refresh_ui()
        except Exception as e:
            print(f"Error in save_game: {e}")

    def load_game(self):
        if not self.root:
            self.root = tk.Tk()
            self.root.withdraw()
            
        try:
            file_path = filedialog.askopenfilename(
                filetypes=[("SOS Game Files", "*.sos"), ("JSON Game Files", "*.json"), ("Text Files", "*.txt"), ("All Files", "*.*")],
                title="Load Game Recording"
            )
            
            if file_path:
                success = self.recorder.load_from_file(file_path)
                if success:
                    print("Game loaded successfully!")
                    self.refresh_ui()
                else:
                    print("Failed to load game")
        except Exception as e:
            print(f"Error in load_game: {e}")

    def replay_game(self):
        if not hasattr(self.recorder, 'moves_history') or not self.recorder.moves_history:
            print("No moves to replay! Record or load a game first.")
            return
        
        print(f"Starting replay of {len(self.recorder.moves_history)} moves...")
        
        pygame.time.set_timer(pygame.USEREVENT, 0)
        self.recorder.is_replaying = True
        
        try:
            self.game.board = [['' for _ in range(self.game.n)] for _ in range(self.game.n)]
            
            if self.game.mode == "General" and hasattr(self.game.game_instance, 'scores'):
                self.game.game_instance.scores = {'Blue': 0, 'Red': 0}
            
            self.refresh_ui()
            pygame.display.flip()
            pygame.time.delay(500)            
            for i, move in enumerate(self.recorder.moves_history):
                row = move['row'] 
                col = move['col']
                symbol = move['symbol']
                player = move['player']
                
                print(f"Replaying move {i+1}/{len(self.recorder.moves_history)}: {player} places {symbol} at ({row},{col})")
                
                self.game.current_player = player
                
                self.game.board[row][col] = (symbol, player)
                
                if self.game.mode == "General" and hasattr(self.game.game_instance, 'check_sos_formation'):
                    self.game.game_instance.check_sos_formation(row, col, player)
                
                self.refresh_ui()
                pygame.display.flip()
                
                pygame.time.delay(800)
                
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        pygame.quit()
                        sys.exit()
            
            print("Replay complete!")
            
        except Exception as e:
            print(f"Error during replay: {e}")
            import traceback
            traceback.print_exc()
        
        finally:
            self.recorder.is_replaying = False
            
            if self.game.player_type[self.game.current_player] == 'computer' and not self.game.game_over:
                pygame.time.set_timer(pygame.USEREVENT, 1000)
            
    def handle_mouse_click(self, pos):
        if self.replay_rect.collidepoint(pos):
            self.replay_game()
            return
            
        if self.input_box.collidepoint(pos):
            self.input_active = True
        else:
            self.input_active = False

        for rect, text, action in self.buttons:
            if rect.collidepoint(pos):
                action()
                return

        if self.game.game_over or self.recorder.is_replaying:
            return

        if (self.start_x <= pos[0] <= self.start_x + self.grid_size and
            self.start_y <= pos[1] <= self.start_y + self.grid_size):
            col = (pos[0] - self.start_x) // self.cell_size
            row = (pos[1] - self.start_y) // self.cell_size
            
            if 0 <= row < self.game.n and 0 <= col < self.game.n:
                if not self.game.board[row][col]:  # Check if cell is empty
                    symbol = self.game.player_symbols[self.game.current_player]
                    player = self.game.current_player
                    
                    if self.game.place_symbol(row, col):
                        if self.recorder.is_recording:
                            self.recorder.record_move(row, col, symbol, player)
                            
                        self.refresh_ui()
                        
                        if self.game.player_type[self.game.current_player] == 'computer' and not self.game.game_over:
                            pygame.time.set_timer(pygame.USEREVENT, 1000)
                        else:
                            pygame.time.set_timer(pygame.USEREVENT, 0)

    def run(self):
        while self.running:
            self.screen.fill(self.WHITE)
            self.draw_sidebar()
            self.draw_grid()
            self.draw_board()
            self.draw_game_status()
            self.draw_replay_button()

            for event in pygame.event.get():
                if event.type == pygame.USEREVENT:
                    if self.game.player_type[self.game.current_player] == 'computer' and not self.game.game_over:
                        symbol = self.game.player_symbols[self.game.current_player]
                        player = self.game.current_player
                        
                        old_board = []
                        for row in self.game.board:
                            old_board.append(row[:])
                        
                        result = self.game.place_symbol(None, None)
                        
                        if result and self.recorder.is_recording:
                            computer_move_row = None
                            computer_move_col = None
                            computer_move_symbol = None
                            
                            for r in range(self.game.n):
                                for c in range(self.game.n):
                                    if (not old_board[r][c] and self.game.board[r][c] and 
                                        self.game.board[r][c][1] == player):
                                        computer_move_row = r
                                        computer_move_col = c
                                        computer_move_symbol = self.game.board[r][c][0]
                                        break
                                if computer_move_row is not None:
                                    break
                            
                            if computer_move_row is not None:
                                print(f"Recording computer move: {player} placed {computer_move_symbol} at ({computer_move_row}, {computer_move_col})")
                                self.recorder.record_move(computer_move_row, computer_move_col, computer_move_symbol, player)
                        
                        self.refresh_ui()
                        
                        if self.game.player_type[self.game.current_player] == 'computer' and not self.game.game_over:
                            pygame.time.set_timer(pygame.USEREVENT, 1000)
                        else:
                            pygame.time.set_timer(pygame.USEREVENT, 0)
                        
                elif event.type == pygame.QUIT:
                    self.running = False
                
                elif event.type == pygame.MOUSEBUTTONDOWN:
                    self.handle_mouse_click(event.pos)
                
                elif event.type == pygame.KEYDOWN:
                    if self.input_active:
                        if event.key == pygame.K_RETURN:
                            self.update_board_size()
                            self.input_active = False
                        elif event.key == pygame.K_BACKSPACE:
                            self.input_text = self.input_text[:-1]
                        elif event.unicode.isdigit():
                            self.input_text += event.unicode

            pygame.display.flip()

        pygame.quit()
        sys.exit()


if __name__ == "__main__":
    pygame.init()
    if not pygame.font.get_init():
        pygame.font.init()
    
    game = SOSGame()
    ui = SOSGameUI(game)
    ui.run()